In [ ]:
"""
Task 3: MPC vs. Day-ahead under Forecast Error
"""

import numpy as np
import cvxpy as cp
import matplotlib.pyplot as plt
import pandas as pd

# ==============================================================================
# Load, generation, and tariffs
# ==============================================================================
def read_load_generation(load_file, pv_file):
    """
    Read load and generation from CSV files
    """
    df_load = pd.read_csv(load_file)
    df_pv = pd.read_csv(pv_file)

    dates = df_load.iloc[:, 0].astype(str).tolist()

    load_profiles = df_load.iloc[:, 1:].values.flatten()
    pv_profiles = df_pv.iloc[:, 1:].values.flatten()
    days = len(df_load)

    return load_profiles, pv_profiles, days, dates


def generate_eta_profiles(days, steps_per_day):
    """
    Generates the daily time-of-use tariff and copies it
    across the entire multi-day horizon.
    """
    eta_day = np.zeros(steps_per_day)
    eta_day[np.r_[0:14, 44:48]] = 0.03   # Off-peak rate ($/kWh)
    eta_day[np.r_[14:28, 40:44]] = 0.06  # Shoulder rate ($/kWh)
    eta_day[28:40]               = 0.30  # Peak rate ($/kWh)

    eta_profiles = np.tile(eta_day, days)
    return eta_profiles

# ==============================================================================
# Core algorithmic function definitions: MPC QP
# ==============================================================================
def solve_mpc_step(p_load_pred, p_pv_pred, eta_pred, z_current, N, delta, p_min, p_max, c_max, w):
    x1 = cp.Variable(N)
    x2 = cp.Variable(N)
    z  = cp.Variable(N+1)

    # =========================================================================
    # @TODO: Paste your completed objective from MPC Task 1 here:
    #   min sum_k [ -delta * eta_pred(k) * x1(k) + w * eta_pred(k) * (x2(k))^2 ]
    # =========================================================================

    objective = None

    constraints = [
        x2 == p_load_pred - p_pv_pred - x1,
        x1 >= p_min, x1 <= p_max,
        z >= 0.0, z <= c_max,
        z[0] == z_current
    ]
    # =========================================================================
    # Battery-energy dynamics: LaTeX indexing versus Python indexing
    # -------------------------------------------------------------------------
    # The mathematical MPC model uses horizon steps k = 1,...,N:
    #
    #     z(t+k|t) = z(t+k-1|t) - delta*x1(t+k|t)
    #
    # with:
    #
    #     z(t|t)   = measured battery energy at the current time, and
    #     z(t+N|t) = z(t|t) for the cyclic terminal condition.
    #
    # Python arrays use zero-based indexing, and the state vector contains
    # one more element than the dispatch vector:
    #
    #     z[0],...,z[N]
    #         <-> z(t|t), z(t+1|t),...,z(t+N|t)
    #
    #     x1[0],...,x1[N-1]
    #         <-> x1(t+1|t),...,x1(t+N|t)
    #
    # Therefore:
    #
    #     z[k+1] = z[k] - delta*x1[k],    k = 0,...,N-1
    #
    # This is the same recurrence as the LaTeX equation, expressed using
    # Python's zero-based indexing. The N+1 state values represent the
    # current state plus one boundary state after each of the N dispatch steps.
    # =========================================================================
    for k in range(N):
        constraints += [z[k+1] == z[k] - delta * x1[k]]

    constraints += [z[N] == z[0]]

    prob = cp.Problem(objective, constraints)
    prob.solve(solver=cp.OSQP)

    if prob.status != cp.OPTIMAL:
        print(f"Warning: MPC subproblem failed to solve. Status: {prob.status}")

    return x1.value, x2.value, z.value


def run_mpc_loop(sim_steps, N_pred, delta, forecast_load, forecast_pv, actual_load, actual_pv, eta_profiles, p_min, p_max, c_max, z_init, w):
    """
    Executes the receding-horizon mechanism: predicts from the forecast data,
    solves, executes the 1st step against the actual data, and advances the
    system state forward.
    """
    executed_x1_BESS  = np.zeros(sim_steps)
    executed_x2_actual = np.zeros(sim_steps)
    executed_z_SOC    = np.zeros(sim_steps)

    current_z = z_init

    for t in range(sim_steps):
        p_load_pred = forecast_load[t : t + N_pred]
        p_pv_pred   = forecast_pv[t : t + N_pred]
        eta_pred    = eta_profiles[t : t + N_pred]

        x1_pred, _, _ = solve_mpc_step(
            p_load_pred, p_pv_pred, eta_pred, current_z, N_pred, delta, p_min, p_max, c_max, w
        )

        x1_exec_t = x1_pred[0]
        x2_actual_t = actual_load[t] - actual_pv[t] - x1_exec_t

        executed_x1_BESS[t]   = x1_exec_t
        executed_x2_actual[t] = x2_actual_t
        executed_z_SOC[t]     = current_z

        current_z = current_z - delta * x1_exec_t

    return executed_x1_BESS, executed_x2_actual, executed_z_SOC


In [ ]:
"""
Main Execution
"""

steps_per_day = 48
delta = 0.5

perfect_load, perfect_pv, total_days, dates = read_load_generation('perfect_load.csv', 'perfect_pv.csv')
synthetic_load, synthetic_pv, _, _ = read_load_generation('synthetic_load.csv', 'synthetic_pv.csv')

eta_profiles = generate_eta_profiles(total_days, steps_per_day)

pred_days = 1
N_pred = pred_days * steps_per_day
sim_days = total_days - pred_days
sim_steps = sim_days * steps_per_day

total_customers = 1330
p_min = -5.0 * total_customers
p_max = 5.0 * total_customers
c_max = 10.0 * total_customers
z_init = 5.0 * total_customers
w = 1.0

time_indices_sim = np.arange(sim_steps)
timeline_sim = time_indices_sim * delta

x1_mpc_actual, x2_mpc_actual, z_mpc_actual = run_mpc_loop(
    sim_steps, N_pred, delta,
    synthetic_load, synthetic_pv,
    perfect_load, perfect_pv,
    eta_profiles, p_min, p_max, c_max, z_init, w
)
bill_mpc_actual = np.sum(eta_profiles[:sim_steps] * x2_mpc_actual * delta)

x1_mpc_ideal, x2_mpc_ideal, z_mpc_ideal = run_mpc_loop(
    sim_steps, N_pred, delta,
    perfect_load, perfect_pv,
    perfect_load, perfect_pv,
    eta_profiles, p_min, p_max, c_max, z_init, w
)
bill_mpc_ideal = np.sum(eta_profiles[:sim_steps] * x2_mpc_ideal * delta)

### NOTE: fill in the Day-ahead Task 2 numbers you obtained for your own
### data/run below, so the printed comparison reflects YOUR results.
day_ahead_bill_ideal = 4537.07
day_ahead_bill_actual = 6479.08

print("=" * 60)
print(f"Day-ahead, perfect forecast   -- Bill: ${day_ahead_bill_ideal:.2f}")
print(f"Day-ahead, synthetic forecast -- Bill: ${day_ahead_bill_actual:.2f}")
print(f"MPC, perfect forecast          -- Bill: ${bill_mpc_ideal:.2f}")
print(f"MPC, synthetic forecast        -- Bill: ${bill_mpc_actual:.2f}")
print("-" * 60)
print(f"Day-ahead financial loss from forecast error: ${day_ahead_bill_actual - day_ahead_bill_ideal:.2f}")
print(f"MPC financial loss from forecast error:        ${bill_mpc_actual - bill_mpc_ideal:.2f}")
print("=" * 60)

plt.rcParams.update({'font.size': 13, 'axes.labelsize': 14, 'axes.titlesize': 15, 'savefig.dpi': 300})

total_hours_sim = timeline_sim[-1] + delta
tick_hours_sim = np.arange(0, total_hours_sim + 1, 6)
time_labels_sim = []
for h in tick_hours_sim:
    day_idx = int(h // 24)
    time_str = f"{int(h % 24):02d}:30"
    if h % 24 == 0 and day_idx < sim_days:
        time_labels_sim.append(f"{dates[day_idx]}\n{time_str}")
    else:
        time_labels_sim.append(f"\n{time_str}")

plt.figure(figsize=(14, 5))
plt.plot(timeline_sim, x1_mpc_actual, color='tab:red', linewidth=1.6, label='MPC, synthetic forecast')
plt.plot(timeline_sim, x1_mpc_ideal, color='tab:green', linewidth=1.6, label='MPC, perfect forecast')
for d in range(1, sim_days + 1):
    plt.axvline(x=d * 24.0, color='red', linestyle=':', linewidth=1.5)
plt.title('Executed BESS Power: Synthetic vs. Perfect Forecast')
plt.xlabel('Time Horizon')
plt.ylabel('BESS Power (kW)')
plt.xlim(0, total_hours_sim)
plt.xticks(tick_hours_sim, time_labels_sim, rotation=45)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='upper right', frameon=True, facecolor='white', edgecolor='none')
plt.tight_layout()
plt.show()

plt.figure(figsize=(14, 5))
plt.plot(timeline_sim, x2_mpc_actual, color='tab:red', linewidth=1.6, label='MPC, synthetic forecast')
plt.plot(timeline_sim, x2_mpc_ideal, color='tab:green', linewidth=1.6, label='MPC, perfect forecast')
for d in range(1, sim_days + 1):
    plt.axvline(x=d * 24.0, color='red', linestyle=':', linewidth=1.5)
plt.title('Realized Grid Power: Synthetic vs. Perfect Forecast')
plt.xlabel('Time Horizon')
plt.ylabel('Grid Power $x_2$ (kW)')
plt.xlim(0, total_hours_sim)
plt.xticks(tick_hours_sim, time_labels_sim, rotation=45)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='upper right', frameon=True, facecolor='white', edgecolor='none')
plt.tight_layout()
plt.savefig('mpc_vs_dayahead_grid_power.png', dpi=300)
plt.show()

